# Simulación: Impacto de Comisiones en FPV vs Inversión Propia (ETFs)

**Escenario base**
- Aporte mensual: $1.000.000 COP
- Horizonte: 15 años
- Rentabilidad bruta esperada: 15% anual
- Comisión FPV: 3.5% anual sobre saldo
- Comisión ETFs (eToro / BVC): 0.05% anual

Este notebook calcula el valor futuro, muestra el impacto de las comisiones y compara el resultado neto después de impuestos al momento del retiro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Parámetros
monthly_contribution = 1_000_000
years = 15
months = years * 12
gross_annual_return = 0.15
fee_fpv = 0.035
fee_etf = 0.0005
tax_rate_etoro = 0.15  # Ganancia ocasional

def future_value(monthly, annual_rate, years):
    monthly_rate = annual_rate / 12
    n = years * 12
    return monthly * (((1 + monthly_rate)**n - 1) / monthly_rate)

net_fpv = gross_annual_return - fee_fpv
net_etf = gross_annual_return - fee_etf

fv_fpv = future_value(monthly_contribution, net_fpv, years)
fv_etf = future_value(monthly_contribution, net_etf, years)
total_contributed = monthly_contribution * months
gains_etf = fv_etf - total_contributed
tax_etoro = gains_etf * tax_rate_etoro
fv_etoro_net = fv_etf - tax_etoro
fv_bvc_net = fv_etf  # Exento si aplica Art. 36-1

print(f"Total aportado: ${total_contributed:,.0f}")
print(f"Valor final FPV: ${fv_fpv:,.0f}")
print(f"Valor final ETFs (bruto): ${fv_etf:,.0f}")
print(f"Impuesto eToro (15%): ${tax_etoro:,.0f}")
print(f"Valor neto eToro: ${fv_etoro_net:,.0f}")
print(f"Valor neto BVC (exento): ${fv_bvc_net:,.0f}")

In [ ]:
# Gráfico de evolución del capital y crecimiento del costo de la comisión
years_array = np.arange(0, years + 1)

def cumulative_fv(monthly, annual_rate, max_years):
    values = []
    for y in range(max_years + 1):
        if y == 0:
            values.append(0)
        else:
            values.append(future_value(monthly, annual_rate, y))
    return np.array(values)

fpv_path = cumulative_fv(monthly_contribution, net_fpv, years)
etf_path = cumulative_fv(monthly_contribution, net_etf, years)
no_fee_path = cumulative_fv(monthly_contribution, gross_annual_return, years)

fee_drag = no_fee_path - fpv_path  # Costo acumulado aproximado de la comisión 3.5%

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: evolución del capital
ax1.plot(years_array, fpv_path/1e6, label='FPV (3.5%)', linewidth=2.5, color='#e74c3c')
ax1.plot(years_array, etf_path/1e6, label='ETFs (0.05%)', linewidth=2.5, color='#27ae60')
ax1.fill_between(years_array, fpv_path/1e6, etf_path/1e6, alpha=0.15, color='gray')
ax1.set_title('Evolución del Capital')
ax1.set_xlabel('Años')
ax1.set_ylabel('Millones de COP')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Derecha: crecimiento del costo de la comisión 3.5%
ax2.plot(years_array, fee_drag/1e6, linewidth=2.5, color='#c0392b')
ax2.fill_between(years_array, 0, fee_drag/1e6, alpha=0.3, color='#e74c3c')
ax2.set_title('Costo Acumulado de la Comisión 3.5%\n(vs no cobrar comisión)')
ax2.set_xlabel('Años')
ax2.set_ylabel('Millones de COP')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nCosto acumulado aproximado de la comisión 3.5% a 15 años: ${fee_drag[-1]:,.0f}")

In [ ]:
# Gráfico comparativo de valor neto final
scenarios = ['FPV\n(3.5% comisión)', 'eToro\n(después de 15%)', 'BVC\n(exento Art. 36-1)']
values = [fv_fpv / 1e6, fv_etoro_net / 1e6, fv_bvc_net / 1e6]
colors = ['#e74c3c', '#f39c12', '#27ae60']

plt.figure(figsize=(9, 6))
bars = plt.bar(scenarios, values, color=colors, edgecolor='black', linewidth=0.8)
plt.title('Valor Neto Final a 15 Años\n(después de comisiones e impuestos al retiro)', fontsize=14)
plt.ylabel('Millones de COP')
plt.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'${val:.0f}M', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Tabla de tratamiento tributario al retiro
tax_treatment = pd.DataFrame({
    'Vehículo': ['FPV', 'BVC (acciones listadas)', 'eToro / extranjero'],
    'Tratamiento de ganancias al retirar': [
        'Exento (capital y rendimientos)',
        'Exento (no es renta ni ganancia ocasional)',
        'Ganancia ocasional al 15%'
    ],
    'Condición': [
        '10 años, pensión, vivienda, etc.',
        'No vender >3% de acciones en circulación (Art. 36-1)',
        'Mantener ≥2 años (activo fijo)'
    ]
})
print(tax_treatment.to_string(index=False))

## Conclusiones

- La comisión del **3.5%** del FPV se cobra sobre el saldo y crece de forma significativa con el tiempo.
- Incluso pagando el 15% de ganancia ocasional en eToro, el resultado neto suele ser superior al FPV con comisiones altas.
- La BVC puede ser aún más eficiente tributariamente para acciones colombianas (potencialmente exento).